# Hands-On Prep

**[website version](https://training.nrp-nautilus.io/cms-hats/3_prep.html)** — run cells with **Shift+Enter**.

In this section we will prepare to run the hands-on exercise.

**On the NRP USCMS Analysis Hub**, `nbgitpuller` already cloned the repo into your home directory when you opened this notebook — you don't need the `git clone` step from the website version. Start at the username cell below.

## What you're building: a jet classifier

At the LHC, quarks and gluons produced in a collision don't fly out as free particles — they fragment and hadronize into a collimated spray of particles called a **jet**. Different progenitors leave different fingerprints on that spray: gluon and light-quark jets are mostly unstructured, while jets seeded by a boosted **W**, **Z**, or top quark carry visible substructure (at least one "prong") from the heavier particle's decay products landing inside a single jet. Telling these apart — **jet tagging** — is a standard part of many LHC analyses, e.g. finding boosted W/Z bosons or top quarks in high-energy events.

This exercise trains a classifier to do exactly that, using the [`hls4ml_lhc_jets_hlf`](https://www.openml.org/search?type=data&id=42468) dataset: roughly 830,000 simulated jets, each labeled as one of five classes — gluon (**g**), light quark (**q**), **W**, **Z**, or top quark (**t**) — and described by 16 high-level features (energy correlation functions, jet mass, particle multiplicity, and related substructure variables) rather than raw detector images. That keeps the input small enough to train a plain fully-connected neural network directly on tabular data — no convolutions, no jet images.

`jet_class.py` scales the network up from a teaching-sized 64→32→32 model to `4096→4096→2048→1024` and trains with mixed precision on a GPU, so you'll see a real (if short) GPU training job rather than a CPU toy. `analyze_jet_class.py` then evaluates the trained model: accuracy, a confusion matrix, ROC curves, and feature-distribution plots.

This material is adapted from [Javier Duarte's PHYS 139/239 course at UCSD](https://jduarte.physics.ucsd.edu/phys139_239/03_Tabular_Data_NN.html), which covers the same exercise — and its extensions, like regularization, learning rate, and optimizer choice — in much more depth if you want to go further.

## ⚙️ Set your username

In [ ]:
export USER=changeme   # ✏️ EDIT to your short name, then Shift+Enter
cd ~/cms-hats/workspace
if [ "$USER" = changeme ]; then echo "⚠️  Edit USER above first, then re-run"; fi


## Create a shared PVC for the training

The hands-on jobs write their outputs to one persistent volume claim (PVC) for the whole training. You only need to create this PVC once.

`yamls/pvc.yaml`:

```yaml
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: cms-nrp-hats-<username>
  namespace: us-cms
spec:
  storageClassName: rook-ceph-block
  accessModes:
  - ReadWriteOnce
  resources:
    requests:
      storage: 5Gi
```

In [ ]:
cp yamls/pvc.yaml /tmp/cms-nrp-hats-pvc-${USER}.yaml
perl -pi -e 's/<username>/$ENV{USER}/g' /tmp/cms-nrp-hats-pvc-${USER}.yaml


In [ ]:
kubectl apply -n us-cms -f /tmp/cms-nrp-hats-pvc-${USER}.yaml


In [ ]:
kubectl get pvc -n us-cms cms-nrp-hats-${USER}


## The container image

Kubernetes cannot use a local unnamed Docker image, so the image needs a registry name the cluster can pull. Use the prepared image for this training:

In [ ]:
export IMAGE=ghcr.io/ddiaz006/cms-hats-jet-class:0.2


**Optional — build your own image instead.** This is a **🖥️ Terminal step** (Docker build/push aren't notebook-cell operations) and only needed if you don't want to use the prepared image:

```bash
export IMAGE=ghcr.io/<github-user-or-org>/cms-hats-jet-class:0.2
docker build --platform linux/amd64 -f code/Dockerfile.jet-class -t "$IMAGE" .
docker push "$IMAGE"
```

If the image is hosted on GHCR, make sure the package is public — a private image shows up as `ImagePullBackOff` with a `401 Unauthorized` message.

## Prepare the YAML

Make a temporary copy of the training manifest and fill in the placeholders:

`yamls/jet-class-job.yaml`:

```yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: jet-class-<username>
  namespace: us-cms
spec:
  template:
    spec:
      restartPolicy: Never
      securityContext:
        runAsUser: 1000
        runAsGroup: 100
        fsGroup: 100
        fsGroupChangePolicy: OnRootMismatch
      containers:
      - name: jet-class
        image: <YOUR_IMAGE>
        env:
        - name: RUN_ID
          value: single
        - name: OUTPUT_DIR
          value: /training/jet-class
        - name: EPOCHS
          value: "50"
        - name: BATCH_SIZE
          value: "8192"
        - name: MODEL_WIDTHS
          value: "4096,4096,2048,1024"
        - name: MIXED_PRECISION
          value: "1"
        resources:
          requests:
            cpu: "4"
            memory: 8Gi
            nvidia.com/gpu: 1
          limits:
            cpu: "4"
            memory: 8Gi
            nvidia.com/gpu: 1
        volumeMounts:
        - name: training-storage
          mountPath: /training

      volumes:
      - name: training-storage
        persistentVolumeClaim:
          claimName: cms-nrp-hats-<username>
```

In [ ]:
cd ~/cms-hats/workspace
cp yamls/jet-class-job.yaml /tmp/jet-class-${USER}.yaml
perl -pi -e 's/<username>/$ENV{USER}/g; s|<YOUR_IMAGE>|$ENV{IMAGE}|g' /tmp/jet-class-${USER}.yaml


The manifest creates the GPU training Job, mounts the shared PVC at `/training`, and writes jet classifier outputs under `/training/jet-class`.

Do the same for the CPU analysis manifest you'll use once training finishes — preparing both now, while `$USER` and `$IMAGE` are set, means the next episode is just `kubectl apply`.

`yamls/jet-class-analysis-job.yaml`:

```yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: jet-class-analysis-<username>
  namespace: us-cms
spec:
  template:
    spec:
      restartPolicy: Never
      securityContext:
        runAsUser: 1000
        runAsGroup: 100
        fsGroup: 100
        fsGroupChangePolicy: OnRootMismatch
      containers:
      - name: jet-class-analysis
        image: <YOUR_IMAGE>
        command: ["python", "/workspace/analyze_jet_class.py"]
        env:
        - name: RUN_ID
          value: single
        - name: OUTPUT_DIR
          value: /training/jet-class
        resources:
          requests:
            cpu: "2"
            memory: 4Gi
          limits:
            cpu: "2"
            memory: 4Gi
        volumeMounts:
        - name: training-storage
          mountPath: /training

      volumes:
      - name: training-storage
        persistentVolumeClaim:
          claimName: cms-nrp-hats-<username>
```

In [ ]:
cp yamls/jet-class-analysis-job.yaml /tmp/jet-class-analysis-${USER}.yaml
perl -pi -e 's/<username>/$ENV{USER}/g; s|<YOUR_IMAGE>|$ENV{IMAGE}|g' /tmp/jet-class-analysis-${USER}.yaml


Both job manifests are now ready in `/tmp`. Continue in the next episode, [Hands-On Exercise](4_hands_on.ipynb).

---

## ✅ Check your work

Verifies the state of your resources on the cluster — rerun any time.

In [ ]:
bash check.sh 3
